# Importing pyspark and dataset 

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Flight Delay Intelligence").getOrCreate()

flights = spark.read.csv(r"C:\prerit\Projects\Flight-Delay-Intelligence-System\data\raw\flights.csv", header=True, inferSchema=True)

In [6]:
import pandas as pd 
pd.set_option("display.max_columns", None)

In [7]:
flights.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: integer (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = true)
 |-- ARRIVAL_DELAY: integer (null

In [8]:
flights.count()

5819079

In [12]:
flights.groupBy("CANCELLED").count().show()

+---------+-------+
|CANCELLED|  count|
+---------+-------+
|        1|  89884|
|        0|5729195|
+---------+-------+



### Removing cancelled flights

In [16]:
flights = flights.filter(flights.CANCELLED == 0)

In [17]:
flights.groupBy("CANCELLED").count().show()

+---------+-------+
|CANCELLED|  count|
+---------+-------+
|        0|5729195|
+---------+-------+



In [23]:
from pyspark.sql.functions import col, count, when

flights.select([count(when(col(c).isNull(), c)).alias(c) for c in flights.columns]).show()

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-

### Removing null values from ARRIVAL_DELAY, DEPARTURE_DELAY

In [22]:
flights = flights.dropna(subset=["ARRIVAL_DELAY", "DEPARTURE_DELAY"])

In [24]:
flights.count()

5714008

### Creating delay categories

In [25]:
from pyspark.sql.functions import when

flights = flights.withColumn(
    "delay_category",
    when(flights.ARRIVAL_DELAY <= 0, "On Time")
    .when(flights.ARRIVAL_DELAY <= 15, "Minor Delay")
    .when(flights.ARRIVAL_DELAY <= 60, "Moderate Delay")
    .otherwise("Severe Delay")
)

In [26]:
flights.show(5)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+--------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|delay_category|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------

In [27]:
flights.groupBy("delay_category").count().show()

+--------------+-------+
|delay_category|  count|
+--------------+-------+
|       On Time|3627112|
|   Minor Delay|1063398|
|Moderate Delay| 704406|
|  Severe Delay| 319092|
+--------------+-------+

